In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# LOAD DATA
# =========================
train_ds = keras.utils.image_dataset_from_directory(
    r"F:\projects gam3a\Smart-Waste-Classification-NN-PROJECT-\data_processed\train",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    r"F:\projects gam3a\Smart-Waste-Classification-NN-PROJECT-\data_processed\val",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
np.save("classes_cnn.npy", class_names)
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =========================
# CUSTOM CNN MODEL
# =========================
model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    # Normalize pixel values to [0, 1]
    layers.Rescaling(1./255),

    # Data Augmentation
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    # ---- Block 1 ----
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # ---- Block 2 ----
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # ---- Block 3 ----
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # ---- Classifier ----
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,  # ✅ 15 epochs with early stopping
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=4,
            restore_best_weights=True,
            verbose=1
        )
    ]
)

model.save("waste_classifier_custom_cnn.keras")
print("✅ Custom CNN Model saved")

# hoa sha8al bs 34an 3mlt stop bs w 3awz azod el epochs


Found 7036 files belonging to 6 classes.
Using 5629 files for training.
Found 374 files belonging to 6 classes.
Using 74 files for validation.


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 random_flip (RandomFlip)    (None, 224, 224, 3)       0         
                                                                 
 random_rotation (RandomRot  (None, 224, 224, 3)       0         
 ation)                                                          
                                                                 
 random_zoom (RandomZoom)    (None, 224, 224, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 224, 224, 32)      896       
                                          

In [7]:
import tensorflow as tf
import numpy as np

# =========================
# LOAD BOTH MODELS
# =========================
mobile_model = tf.keras.models.load_model("waste_classifier_v2.keras")
custom_model = tf.keras.models.load_model("waste_classifier_custom_cnn.keras")

# Load val_ds again for evaluation
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

val_ds = tf.keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\val",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

# =========================
# EVALUATION
# =========================
print("Evaluating MobileNetV2...")
mobile_loss, mobile_acc = mobile_model.evaluate(val_ds, verbose=0)

print("Evaluating Custom CNN...")
custom_loss, custom_acc = custom_model.evaluate(val_ds, verbose=0)

# =========================
# COMPARISON
# =========================
print("\n========== MODEL COMPARISON ==========")
print(f"{'Model':<20} {'Accuracy':>10} {'Loss':>10}")
print("-" * 42)
print(f"{'MobileNetV2':<20} {mobile_acc:>10.4f} {mobile_loss:>10.4f}")
print(f"{'Custom CNN':<20} {custom_acc:>10.4f} {custom_loss:>10.4f}")
print("-" * 42)

winner = "MobileNetV2" if mobile_acc > custom_acc else "Custom CNN"
print(f"\n🏆 Better Model: {winner}")

Found 300 files belonging to 4 classes.
Using 60 files for validation.
Evaluating MobileNetV2...
Evaluating Custom CNN...

========== MODEL COMPARISON ==========
Model                  Accuracy       Loss
------------------------------------------
MobileNetV2              1.0000     0.0405
Custom CNN               0.2667     1.4725
------------------------------------------

🏆 Better Model: MobileNetV2
